# 03 · Geração fundamentada (item 04)
RAG híbrido + `qwen2.5:3b` via Ollama, com JSON validado por Pydantic, nova tentativa com instrução de correção, recusa controlada, filtro de citações e salvaguarda contra "fique em casa".

Requer o Ollama rodando com `qwen2.5:3b`.

In [1]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from assistente.config import CONFIG

In [2]:
from assistente.ingest import carregar
from assistente.retriever import Buscador
from assistente.rag import Assistente

buscador = Buscador(carregar()).carregar()
assistente = Assistente(buscador)
print("top_k =", assistente.k, "| limiar de recusa =", assistente.limiar)

top_k = 5 | limiar de recusa = 0.84


## Respostas fundamentadas
Cada resposta traz `fontes` (doc_id e página), o aviso de emergência em `o_que_fazer` e `confianca`. Em `meta` aparecem os trechos recuperados e a similaridade máxima usada no limiar.

In [3]:
def mostrar(pergunta, resp, meta):
    print("PERGUNTA:", pergunta)
    print(resp.model_dump_json(indent=2))
    print("score_max:", meta.get("score_max"), "| recuperados:", [r["chunk_id"] for r in meta.get("recuperados", [])],
          "| salvaguarda:", meta.get("salvaguarda"), f"| {meta.get('latencia_s', 0):.1f}s\n")

for q in ["A porta da minha casa começou a emperrar. Isso é sinal de quê?",
          "Com que frequência a Defesa Civil de Santos mede a chuva no Alerta Máximo?",
          "Minha casa vai ser removida por estar em área de risco. A prefeitura precisa me mostrar algum documento antes?"]:
    mostrar(q, *assistente.rag(q))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

PERGUNTA: A porta da minha casa começou a emperrar. Isso é sinal de quê?
{
  "resposta": "Isso pode ser um sinal de que a sua casa está em um local de risco geológico, como um morro. Procure um local seguro e entre em contato com a Defesa Civil.",
  "o_que_fazer": [
    "Em emergência ligue 199 (Defesa Civil)"
  ],
  "fontes": [
    {
      "doc_id": "cemaden_educacao",
      "pagina": 8,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 0.9
}
score_max: 0.8402 | recuperados: ['cemaden_educacao_0006', 'ppdc_santos_0010', 'cemaden_educacao_0004', 'cemaden_educacao_0003', 'cemaden_educacao_0007'] | salvaguarda: False | 3.9s



PERGUNTA: Com que frequência a Defesa Civil de Santos mede a chuva no Alerta Máximo?
{
  "resposta": "A Defesa Civil de Santos mede a chuva de 1 em 1 hora no Alerta Máximo.",
  "o_que_fazer": [
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "ppdc_santos",
      "pagina": 2,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 1.0
}
score_max: 0.8447 | recuperados: ['ppdc_santos_0008', 'cemaden_educacao_0002', 'cemaden_educacao_0025', 'cemaden_educacao_0015', 'ppdc_santos_0002'] | salvaguarda: False | 3.8s



PERGUNTA: Minha casa vai ser removida por estar em área de risco. A prefeitura precisa me mostrar algum documento antes?
{
  "resposta": "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros). Em caso de remoção, a prefeitura precisa fornecer um documento que comprove a remoção e a necessidade de abrigo. Procure um local seguro e entre em contato com a Defesa Civil.",
  "o_que_fazer": [
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "ipt_mapeamento_riscos",
      "pagina": 6,
      "trecho": null
    },
    {
      "doc_id": "ppdc_santos",
      "pagina": 2,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 0.8
}
score_max: 0.8673 | recuperados: ['lei_12608_2012_0019', 'cemaden_educacao_0022', 'lei_12608_2012_0018', 'ipt_mapeamento_riscos_0008', 'ppdc_santos_0004'] | salvaguarda: False | 4.7s



## Recusa em duas camadas
1. **Limiar** (similaridade < 0,84): recusa sem chamar o LLM, o que vale para perguntas fora do tema.
2. **Prompt** ("use apenas os trechos"): recusa perguntas do tema cujo fato não está na base.

In [4]:
for q in ["Qual o melhor time de futebol do Brasil?",
          "Quantas pessoas morreram nos deslizamentos de Petrópolis em 2022?"]:
    mostrar(q, *assistente.rag(q))

PERGUNTA: Qual o melhor time de futebol do Brasil?
{
  "resposta": "A informação não consta na base oficial do assistente. Procure a Defesa Civil do seu município.",
  "o_que_fazer": [
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [],
  "recusa": true,
  "confianca": 0.0
}
score_max: 0.7875 | recuperados: ['cemaden_educacao_0017', 'cemaden_educacao_0028', 'lei_12608_2012_0003', 'cemaden_educacao_0026', 'ipt_mapeamento_riscos_0001'] | salvaguarda: None | 0.0s



PERGUNTA: Quantas pessoas morreram nos deslizamentos de Petrópolis em 2022?
{
  "resposta": "recusa",
  "o_que_fazer": [
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "cemaden_educacao",
      "pagina": 41,
      "trecho": null
    }
  ],
  "recusa": true,
  "confianca": 0.0
}
score_max: 0.8509 | recuperados: ['cemaden_educacao_0015', 'cemaden_educacao_0021', 'cemaden_educacao_0028', 'cemaden_educacao_0016', 'cemaden_educacao_0023'] | salvaguarda: False | 6.8s



## Salvaguarda de segurança (erro nº 1)
Para a pergunta abaixo, a busca não traz o trecho da cartilha que diz "saia imediatamente". Sem esse trecho, o modelo varia entre execuções:
- às vezes diz que não há informação suficiente;
- às vezes orienta **"não saia de casa"**. Isso aconteceu nas 3 execuções do teste de bancada e no placar v2.

A célula a seguir mostra a resposta desta execução.

In [5]:
q = "Choveu a noite toda e apareceu uma rachadura nova na parede. Devo sair de casa?"
mostrar(q, *assistente.rag(q))

PERGUNTA: Choveu a noite toda e apareceu uma rachadura nova na parede. Devo sair de casa?
{
  "resposta": "Diante de sinais de perigo, como rachaduras, trincas ou inclinação de muros, postes e árvores, saia imediatamente do local e procure abrigo em um lugar sem perigo de deslizamento. Depois, informe a Defesa Civil pelo 199.",
  "o_que_fazer": [
    "Saia imediatamente do local",
    "Procure abrigo em lugar sem perigo de deslizamento",
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "cartilha_dc_sp",
      "pagina": 1,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 0.5
}
score_max: 0.8449 | recuperados: ['cemaden_educacao_0004', 'cemaden_educacao_0015', 'cemaden_educacao_0022', 'cemaden_educacao_0005', 'cemaden_educacao_0014'] | salvaguarda: True | 7.7s



A próxima célula aplica a salvaguarda à resposta **real** que o agente deu a essa pergunta no placar v2 (`avaliacao/resultados/v2/respostas_agente.csv`, q017). Ela detecta a orientação de permanecer, confere que nenhum trecho recuperado diz isso e troca a resposta pela orientação oficial da cartilha. Esse mecanismo é determinístico: não depende do modelo.

In [6]:
import pandas as pd
from assistente.rag import salvaguarda_permanencia
from assistente.schemas import RespostaAssistente

v2 = pd.read_csv(CONFIG.dir_resultados / "v2" / "respostas_agente.csv").set_index("id")
print("Resposta do agente no placar v2 (q017):", v2.loc["q017", "resposta"])

errada = RespostaAssistente(resposta=v2.loc["q017", "resposta"], fontes=[{"doc_id": "cemaden_educacao"}], confianca=0.9)
trechos = [r.trecho.texto for r in buscador.buscar(v2.loc["q017", "pergunta"])]
corrigida, acionada = salvaguarda_permanencia(errada, trechos)
print("salvaguarda acionada:", acionada)
print(corrigida.model_dump_json(indent=2))

Resposta do agente no placar v2 (q017): Não saia de casa. Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros).
salvaguarda acionada: True
{
  "resposta": "Diante de sinais de perigo, como rachaduras, trincas ou inclinação de muros, postes e árvores, saia imediatamente do local e procure abrigo em um lugar sem perigo de deslizamento. Depois, informe a Defesa Civil pelo 199.",
  "o_que_fazer": [
    "Saia imediatamente do local",
    "Procure abrigo em lugar sem perigo de deslizamento",
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "cartilha_dc_sp",
      "pagina": 1,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 0.5
}


## Demonstração de alucinação: o mesmo modelo sem contexto
Sem os trechos, o modelo responde com conhecimento próprio, sem fonte verificável, ou inventa.

In [7]:
for q in ["No Alerta Máximo, o que a Defesa Civil de Santos orienta as famílias a fazer?",
          "Qual o acumulado de chuva em 72 horas usado pelo PPDC para a região de São José dos Campos?",
          "Qual o valor do aluguel social em Manaus?"]:
    resp, meta = assistente.sem_contexto(q)
    print("PERGUNTA:", q); print(resp.model_dump_json(indent=2), "\n")

PERGUNTA: No Alerta Máximo, o que a Defesa Civil de Santos orienta as famílias a fazer?
{
  "resposta": "Fique em casa e siga as orientações da Defesa Civil",
  "o_que_fazer": [
    "Fique em casa",
    "Observe as orientações da Defesa Civil",
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "SantosAlertaMáximo",
      "pagina": 12,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 0.8
} 



PERGUNTA: Qual o acumulado de chuva em 72 horas usado pelo PPDC para a região de São José dos Campos?
{
  "resposta": "Preciso consultar os dados do PPDC para te informar com certeza.",
  "o_que_fazer": [
    "Consultar o site oficial do PPDC",
    "Contatar o setor responsável",
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "PPDC_Registro",
      "pagina": 15,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 0.8
} 



PERGUNTA: Qual o valor do aluguel social em Manaus?
{
  "resposta": "O valor do aluguel social em Manaus pode variar, consulte o site do Governo do Amazonas para informações atualizadas.",
  "o_que_fazer": [
    "Consultar o site do Governo do Amazonas",
    "Em emergência ligue 199 (Defesa Civil) ou 193 (Bombeiros)."
  ],
  "fontes": [
    {
      "doc_id": "governo_amazonas",
      "pagina": 12,
      "trecho": null
    }
  ],
  "recusa": false,
  "confianca": 0.8
} 

